# 🤖 Assistente de Investimentos com RPA e N8N
### Bootcamp Santander 2026 / DIO — Versão Operacional Google Colab (Interface Completa)

Este notebook executa o pipeline ponta a ponta do **Assistente de Investimentos** com:
- **Web Scraping / RPA**: Extração automatizada e resiliente de clientes da base legada.
- **Normalização & Validação**: Parsing monetário robusto BRL (Decimal) e schemas de validação.
- **Motor de Cruzamento**: Regra de negócio (Perfil + Saldo >= Aplicação Mínima).
- **Interface Visual Rica**: Tabelas estilizadas, badges de status, cards de recomendação e barras de progresso (`tqdm`).
- **Integração N8N & Modo Demonstração**: Suporte a envio via Webhook ou execução simulada (*dry-run*) sem dependência externa.

--- 
## 📦 Célula 1 — Instalação de Dependências

In [ ]:
# ============================================================
# ETAPA 1 | INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
!pip install -q beautifulsoup4 requests pandas pydantic tqdm ipywidgets
print("✅ Dependências instaladas com sucesso!")

--- 
## ⚙️ Célula 2 — Imports, Configuração e Estilos Visuais

In [ ]:
# ============================================================
# ETAPA 2 | IMPORTS, CONFIGURAÇÃO E ESTILOS VISUAIS
# ============================================================
import requests
import pandas as pd
import re
import time
import json
from io import StringIO
from pathlib import Path
from decimal import Decimal, InvalidOperation
from IPython.display import display, HTML, Markdown, clear_output
from bs4 import BeautifulSoup

# ----------- Configuração -----------
# ATENÇÃO: Substitua pelos URLs do seu repositório no GitHub Pages ou caminho local
URL_CLIENTES = "https://<seu-usuario>.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
URL_CSV = "https://<seu-usuario>.github.io/dio-lab-assistente-investimentos-rpa-n8n/data.csv"

# Deixe VAZIO ou placeholder para ativar o Modo Demonstração (dry-run)
WEBHOOK_N8N = ""

# ----------- Estilos CSS para o Notebook -----------
display(HTML("""<style>
  .header-azul {
    background: linear-gradient(135deg, #0052d4, #4364f7, #6fb1fc);
    color: white;
    padding: 18px 24px;
    border-radius: 12px;
    font-family: 'Segoe UI', sans-serif;
    font-size: 1.35rem;
    font-weight: 700;
    text-align: center;
    margin: 16px 0;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
  }
  .header-verde {
    background: linear-gradient(135deg, #11998e, #38ef7d);
    color: white;
    padding: 14px 20px;
    border-radius: 10px;
    font-family: 'Segoe UI', sans-serif;
    font-weight: 600;
    margin: 12px 0;
    box-shadow: 0 3px 8px rgba(0,0,0,0.1);
  }
  .badge-sucesso {
    background: #d4edda;
    color: #155724;
    border: 1px solid #c3e6cb;
    padding: 8px 14px;
    border-radius: 8px;
    margin: 6px 0;
    font-family: 'Segoe UI', sans-serif;
    font-weight: 500;
  }
  .badge-aviso {
    background: #fff3cd;
    color: #856404;
    border: 1px solid #ffeeba;
    padding: 8px 14px;
    border-radius: 8px;
    margin: 6px 0;
    font-family: 'Segoe UI', sans-serif;
    font-weight: 500;
  }
  .badge-erro {
    background: #f8d7da;
    color: #721c24;
    border: 1px solid #f5c6cb;
    padding: 8px 14px;
    border-radius: 8px;
    margin: 6px 0;
    font-family: 'Segoe UI', sans-serif;
    font-weight: 500;
  }
  .card-recomendacao {
    border-left: 5px solid #007bff;
    background: #f8f9fa;
    padding: 14px 18px;
    border-radius: 8px;
    margin: 10px 0;
    font-family: 'Segoe UI', sans-serif;
    box-shadow: 0 2px 6px rgba(0,0,0,0.08);
  }
  .perfil-conservador {
    border-left-color: #17a2b8;
    background: #e0f7fa;
  }
  .perfil-moderado {
    border-left-color: #28a745;
    background: #e8f5e9;
  }
  .perfil-arrojado {
    border-left-color: #fd7e14;
    background: #fff3e0;
  }
  table thead th {
    background-color: #343a40 !important;
    color: white !important;
    text-align: center;
  }
</style>"""))
print("🎨 Estilos visuais e configurações carregadas!")

--- 
## 🛠️ Célula 3 — Funções de Estilização e Helpers

In [ ]:
# ============================================================
# ETAPA 3 | FUNÇÕES AUXILIARES PARA VISUALIZAÇÃO
# ============================================================
def estilizar_perfil(perfil: str) -> dict:
    """Retorna classe CSS e ícone por perfil."""
    classes = {
        "conservador": "perfil-conservador",
        "moderado": "perfil-moderado",
        "arrojado": "perfil-arrojado",
    }
    icones = {
        "conservador": "🛡️",
        "moderado": "⚖️",
        "arrojado": "🚀",
    }
    p = str(perfil).lower().strip()
    return {
        "classe": classes.get(p, "perfil-moderado"),
        "icone": icones.get(p, "✅"),
    }

def badge(texto: str, tipo: str = "sucesso") -> HTML:
    """Gera um badge HTML colorido."""
    return HTML(f'<div class="badge-{tipo}">{texto}</div>')

def parse_saldo(valor) -> Decimal:
    """Converte corretamente 'R$ 1.234,56' ou '1.234.567,89' para Decimal."""
    if isinstance(valor, Decimal):
        return valor
    if isinstance(valor, (int, float)):
        return Decimal(str(valor))

    s = str(valor).replace("R$", "").strip()
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    s = re.sub(r"[^\d.-]", "", s)

    try:
        return Decimal(s) if s else Decimal("0")
    except InvalidOperation:
        try:
            return Decimal(str(valor).replace(",", "."))
        except InvalidOperation:
            return Decimal("0")

def formatar_moeda(valor) -> str:
    """Formata Decimal/Float para o padrão monetário brasileiro."""
    try:
        v = float(valor)
        return f"R$ {v:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")
    except Exception:
        return f"R$ {valor}"

print("🛠️ Helpers de formatação e visualização prontos!")

--- 
## 📡 Célula 4 — Extração RPA (com Progresso e Tabela Estilizada)

In [ ]:
# ============================================================
# ETAPA 4 | EXTRAÇÃO DE CLIENTES (RPA COM BEAUTIFULSOUP)
# ============================================================
from tqdm.auto import tqdm

def extrair_clientes(url: str, max_retries: int = 3) -> pd.DataFrame:
    display(HTML('<div class="header-verde">📡 ETAPA 1: Extração de Clientes (RPA)</div>'))
    html_text = ""

    # Fallback para caminho local caso a URL não esteja publicada
    caminhos_locais = [Path("docs/index.html"), Path("../docs/index.html")]
    usar_local = not url or "<" in url or "seu-usuario" in url

    if usar_local:
        for p in caminhos_locais:
            if p.exists():
                display(badge(f"ℹ️ Usando arquivo HTML local: {p}", "aviso"))
                with open(p, "r", encoding="utf-8") as f:
                    html_text = f.read()
                break

    if not html_text:
        for tentativa in range(1, max_retries + 1):
            try:
                resp = requests.get(url, timeout=20)
                resp.raise_for_status()
                resp.encoding = "utf-8"
                html_text = resp.text
                break
            except requests.RequestException as e:
                display(badge(f"⚠️ Tentativa {tentativa}/{max_retries} falhou: {e}", "aviso"))
                if tentativa == max_retries:
                    for p in caminhos_locais:
                        if p.exists():
                            display(badge(f"⚠️ Recorrendo ao arquivo local de contingência: {p}", "aviso"))
                            with open(p, "r", encoding="utf-8") as f:
                                html_text = f.read()
                            break
                    if not html_text:
                        display(badge(f"❌ Não foi possível acessar {url}. Verifique a URL.", "erro"))
                        return pd.DataFrame()
                time.sleep(2 ** (tentativa - 1))

    soup = BeautifulSoup(html_text, "html.parser")
    tabela = soup.find("table")
    if not tabela:
        display(badge("❌ Elemento &lt;table&gt; não encontrado na página.", "erro"))
        return pd.DataFrame()

    linhas_html = tabela.find_all("tr")
    if not linhas_html:
        display(badge("❌ Nenhuma linha encontrada na tabela.", "erro"))
        return pd.DataFrame()

    # Mapeamento e detecção flexível de cabeçalhos
    th_elements = tabela.find_all("th")
    if th_elements:
        raw_headers = [th.get_text(strip=True) for th in th_elements]
    else:
        raw_headers = [td.get_text(strip=True) for td in linhas_html[0].find_all(["th", "td"])]

    headers = []
    for h in raw_headers:
        hc = h.lower().replace(" ", "_").replace("-", "")
        if "saldo" in hc:
            headers.append("saldo")
        elif "perfil" in hc:
            headers.append("perfil")
        elif "email" in hc or "e_mail" in hc:
            headers.append("email")
        elif "nome" in hc:
            headers.append("nome")
        else:
            headers.append(hc)

    body = tabela.find("tbody")
    linhas_dados = body.find_all("tr") if body else (linhas_html[1:] if len(linhas_html) > 1 else linhas_html)

    dados = []
    for row in tqdm(linhas_dados, desc="Extraindo clientes", unit="cliente"):
        cols = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cols) == len(headers) and any(cols):
            dados.append(dict(zip(headers, cols)))

    if not dados:
        display(badge("⚠️ Nenhum cliente com dados válidos foi encontrado.", "aviso"))
        return pd.DataFrame()

    df = pd.DataFrame(dados)

    if "saldo" in df.columns:
        df["saldo"] = df["saldo"].apply(parse_saldo)
    if "perfil" in df.columns:
        df["perfil"] = df["perfil"].str.strip().str.capitalize()

    display(badge(f"✅ {len(df)} clientes extraídos com sucesso via RPA!", "sucesso"))

    # Tabela estilizada para apresentação
    df_exib = df.copy()
    if "saldo" in df_exib.columns:
        df_exib["Saldo em Conta"] = df_exib["saldo"].apply(formatar_moeda)
        df_exib = df_exib.drop(columns=["saldo"])
    if "perfil" in df_exib.columns:
        df_exib["Perfil de Investidor"] = df_exib["perfil"].apply(
            lambda p: f"{estilizar_perfil(p)['icone']} {p}"
        )
        df_exib = df_exib.drop(columns=["perfil"])

    col_renames = {"nome": "Nome do Cliente", "email": "E-mail", "id": "ID"}
    df_exib = df_exib.rename(columns={k: v for k, v in col_renames.items() if k in df_exib.columns})

    display(HTML('<div class="header-azul">👥 Tabela de Clientes Extraídos (RPA)</div>'))
    display(df_exib.style.set_properties(**{"text-align": "center", "font-family": "Segoe UI"})
           .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))
    display(HTML("<br>"))
    return df

--- 
## 📈 Célula 5 — Carregamento do CSV (com Normalização e Remoção de BOM)

In [ ]:
# ============================================================
# ETAPA 5 | CARREGAMENTO DOS INVESTIMENTOS (CSV)
# ============================================================
def carregar_investimentos(url: str) -> pd.DataFrame:
    display(HTML('<div class="header-verde">📈 ETAPA 2: Carregamento do Catálogo de Investimentos (CSV)</div>'))
    texto = ""
    caminhos_locais = [Path("docs/data.csv"), Path("../docs/data.csv")]
    usar_local = not url or "<" in url or "seu-usuario" in url

    if usar_local:
        for p in caminhos_locais:
            if p.exists():
                display(badge(f"ℹ️ Carregando arquivo CSV local: {p}", "aviso"))
                with open(p, "r", encoding="utf-8-sig") as f:
                    texto = f.read()
                break

    if not texto:
        try:
            resp = requests.get(url, timeout=20)
            resp.raise_for_status()
            texto = resp.text
        except Exception as e:
            display(badge(f"⚠️ Falha ao baixar CSV remoto ({e}).", "aviso"))
            for p in caminhos_locais:
                if p.exists():
                    with open(p, "r", encoding="utf-8-sig") as f:
                        texto = f.read()
                    break

    # Fallback seguro com os dados embutidos oficiais do desafio
    if not texto:
        display(badge("ℹ️ Usando catálogo oficial embutido de contingência.", "aviso"))
        df = pd.DataFrame([
            {"perfil": "Conservador", "produto": "Poupança", "minimo": 100, "rentabilidade": "6.2%"},
            {"perfil": "Conservador", "produto": "CDB Liquidez Diária", "minimo": 1000, "rentabilidade": "12.5%"},
            {"perfil": "Conservador", "produto": "Tesouro Selic", "minimo": 500, "rentabilidade": "13.0%"},
            {"perfil": "Moderado", "produto": "CDB Prefixado", "minimo": 1000, "rentabilidade": "14.0%"},
            {"perfil": "Moderado", "produto": "Fundo Multimercado", "minimo": 5000, "rentabilidade": "16.0%"},
            {"perfil": "Moderado", "produto": "Tesouro IPCA+", "minimo": 1000, "rentabilidade": "IPCA+6.0%"},
            {"perfil": "Arrojado", "produto": "Ações ETF", "minimo": 500, "rentabilidade": "Variável"},
            {"perfil": "Arrojado", "produto": "Fundo de Ações", "minimo": 10000, "rentabilidade": "Variável"},
            {"perfil": "Arrojado", "produto": "Criptomoedas", "minimo": 100, "rentabilidade": "Variável"},
        ])
    else:
        texto = texto.lstrip("\ufeff")
        df = pd.read_csv(StringIO(texto))

    df.columns = [c.strip().lower() for c in df.columns]

    if "minimo" in df.columns:
        df["minimo"] = pd.to_numeric(
            df["minimo"].astype(str).str.replace(r"[^\d.-]", "", regex=True),
            errors="coerce"
        ).fillna(0).apply(lambda x: Decimal(str(int(x))))

    if "perfil" in df.columns:
        df["perfil"] = df["perfil"].str.strip().str.capitalize()
    if "rentabilidade" in df.columns:
        df["rentabilidade"] = df["rentabilidade"].astype(str).str.strip()

    display(badge(f"✅ Catálogo carregado: {len(df)} produtos de investimento disponíveis.", "sucesso"))

    df_exib = df.copy()
    df_exib["Aporte Mínimo"] = df_exib["minimo"].apply(formatar_moeda)
    df_exib["Perfil"] = df_exib["perfil"].apply(lambda p: f"{estilizar_perfil(p)['icone']} {p}")
    df_exib = df_exib.rename(columns={"produto": "Produto de Investimento", "rentabilidade": "Rentabilidade Projetada"})
    cols_view = ["Perfil", "Produto de Investimento", "Aporte Mínimo", "Rentabilidade Projetada"]
    df_exib = df_exib[[c for c in cols_view if c in df_exib.columns]]

    display(HTML('<div class="header-azul">📦 Catálogo Oficial de Investimentos</div>'))
    display(df_exib.style.set_properties(**{"text-align": "center", "font-family": "Segoe UI"})
           .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))
    display(HTML("<br>"))
    return df

--- 
## 🔗 Célula 6 — Lógica de Cruzamento (com Cards Visuais por Perfil)

In [ ]:
# ============================================================
# ETAPA 6 | CRUZAMENTO PERFIL + SALDO (REGRA DE NEGÓCIO)
# ============================================================
INTROS = {
    "Conservador": "🛡️ foco em segurança e preservação de capital",
    "Moderado": "⚖️ equilíbrio entre segurança e rentabilidade",
    "Arrojado": "🚀 potencial de retorno elevado com maior exposição a risco",
}

def cruzar_perfil_saldo(df_clientes: pd.DataFrame, df_invest: pd.DataFrame) -> pd.DataFrame:
    display(HTML('<div class="header-verde">🔗 ETAPA 3: Cruzamento de Perfis e Aplicação Mínima</div>'))

    recomendacoes = []
    invest_records = df_invest.to_dict("records")

    for _, cliente in df_clientes.iterrows():
        nome = cliente["nome"]
        perfil = cliente["perfil"]
        saldo = Decimal(str(cliente["saldo"]))
        email = cliente.get("email", "")

        do_perfil = [i for i in invest_records if str(i["perfil"]).lower() == str(perfil).lower()]
        compativeis = sorted(
            [i for i in do_perfil if saldo >= Decimal(str(i["minimo"]))],
            key=lambda x: Decimal(str(x["minimo"]))
        )

        intro = INTROS.get(perfil, "estratégia sob medida")
        saldo_fmt = formatar_moeda(saldo)

        if not compativeis:
            mensagem = (
                f"Olá {nome}, identificamos seu perfil {perfil}, mas seu saldo atual ({saldo_fmt}) "
                f"ainda não atinge o aporte mínimo exigido para as opções disponíveis. "
                f"Recomendamos aguardar ou aportar valores adicionais."
            )
        else:
            produtos_txt = "\n".join(
                f" • {i['produto']} — rentabilidade {i['rentabilidade']} (mín. {formatar_moeda(i['minimo'])})"
                for i in compativeis
            )
            mensagem = (
                f"Olá {nome}, com base no seu perfil {perfil} ({intro}) e saldo de {saldo_fmt}, "
                f"recomendamos:\n{produtos_txt}"
            )

        recomendacoes.append({
            "nome": nome,
            "email": email,
            "perfil": perfil,
            "saldo": saldo,
            "investimentos": compativeis,
            "mensagem": mensagem
        })

    df_rec = pd.DataFrame(recomendacoes)
    display(badge(f"✅ {len(df_rec)} clientes cruzados e recomendações geradas com sucesso!", "sucesso"))

    # ---------- Renderização dos Cards HTML de Recomendação ----------
    display(HTML('<div class="header-azul">💡 Recomendações Personalizadas aos Clientes</div>'))
    for _, r in df_rec.iterrows():
        estilo = estilizar_perfil(r["perfil"])
        classe = estilo["classe"]
        icone = estilo["icone"]

        if r["investimentos"]:
            produtos_html = "<br>".join(
                f'&nbsp;&nbsp;• <strong>{i["produto"]}</strong> — ' 
                f'rentabilidade <em>{i["rentabilidade"]}</em> (mín. {formatar_moeda(i["minimo"])})'
                for i in r["investimentos"]
            )
        else:
            produtos_html = "&nbsp;&nbsp;⚠️ <em>Nenhuma opção compatível com o saldo atual.</em>"

        card = f"""
        <div class="card-recomendacao {classe}">
            <strong>👤 {r['nome']}</strong> — Perfil: {icone} {r['perfil']}<br>
            <span>💰 Saldo Disponível: <strong>{formatar_moeda(r['saldo'])}</strong></span><br>
            <span>📈 <strong>Investimentos Elegíveis:</strong></span><br>
            {produtos_html}<br><br>
            <div style="background: rgba(255,255,255,0.7); padding: 8px 12px; border-radius: 6px;">
                <em>📧 Mensagem Gerada:</em><br>{r['mensagem'].replace(chr(10), '<br>')}
            </div>
        </div>
        """
        display(HTML(card))

    display(HTML("<br>"))
    return df_rec

--- 
## 📤 Célula 7 — Envio ao N8N (com Suporte a Modo Demonstração)

In [ ]:
# ============================================================
# ETAPA 7 | ENVIO AO N8N (MODO PRODUÇÃO / DEMONSTRAÇÃO)
# ============================================================
def enviar_para_n8n(df_rec: pd.DataFrame, webhook_url: str = "", modo_ia: bool = False) -> dict:
    display(HTML('<div class="header-verde">📤 ETAPA 4: Integração com o Orquestrador N8N</div>'))

    # Modo Demonstração (dry-run)
    if not webhook_url or "<" in webhook_url or "seu-n8n" in webhook_url:
        display(badge(
            "ℹ️ MODO DEMONSTRAÇÃO ATIVO — Webhook N8N não configurado ou em modo simulado. "
            "Resultados validados localmente com sucesso!",
            "aviso"
        ))
        return {
            "modo": "demonstracao",
            "status": "sucesso_local",
            "total_clientes": len(df_rec),
            "mensagem": "Dados estruturados prontos para envio ao N8N."
        }

    # Modo Produção
    try:
        payload = {
            "clientes": df_rec[["nome", "perfil", "saldo", "mensagem"]].to_dict("records"),
            "total": len(df_rec),
            "gerado_por": "RPA Colab",
            "modo_ia": modo_ia,
            "timestamp": pd.Timestamp.now().isoformat(),
        }
        # Ajuste de serialização para Decimals no payload
        for c in payload["clientes"]:
            c["saldo"] = float(c["saldo"])

        display(badge("📡 Enviando dados para o Webhook do N8N...", "aviso"))
        resp = requests.post(
            webhook_url,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=60 # Timeout estendido para acomodar nós de IA Generativa
        )
        resp.raise_for_status()
        display(badge(f"✅ Dados recebidos e processados pelo N8N com sucesso! HTTP {resp.status_code}", "sucesso"))

        if resp.text:
            try:
                return resp.json()
            except json.JSONDecodeError:
                return {"resposta_raw": resp.text}
        return {"resposta_raw": "Sucesso (sem corpo de retorno)"}

    except requests.Timeout:
        display(badge("⚠️ Timeout ao aguardar o N8N. O workflow pode estar processando os nós de IA.", "aviso"))
        return {"erro": "timeout"}
    except requests.RequestException as e:
        display(badge(f"❌ Falha na comunicação com o N8N: {e}", "erro"))
        return {"erro": str(e)}

--- 
## 🎛️ Célula 8 — Seleção Interativa de Modo de Execução

In [ ]:
# ============================================================
# ETAPA 8 | SELEÇÃO DE MODO DE EXECUÇÃO (WIDGET)
# ============================================================
from ipywidgets import widgets, Layout

modo_widget = widgets.Dropdown(
    options=[
        ("🛡️ Modo Demonstração (Sem N8N / Local)", "demo"),
        ("🏭 Modo Produção (Com Webhook N8N)", "producao"),
        ("🤖 Modo Produção + IA Generativa (OpenAI/Gemini)", "producao_ia"),
    ],
    value="demo",
    description="Modo:",
    style={"description_width": "initial"},
    layout=Layout(width="460px"),
)

ia_widget = widgets.Checkbox(
    value=False,
    description="Ativar processamento com IA Generativa no N8N",
    disabled=False,
)

display(HTML('<div class="header-azul">🎛️ Configuração de Execução do Pipeline</div>'))
display(modo_widget)
display(ia_widget)
print("\n👉 Selecione as opções acima e execute a Célula 9 para rodar o pipeline completo.")

--- 
## 🚀 Célula 9 — Execução do Pipeline Ponta a Ponta

In [ ]:
# ============================================================
# ETAPA 9 | PIPELINE PRINCIPAL — EXECUÇÃO PONTA A PONTA
# ============================================================
from IPython.display import clear_output
from tqdm.auto import tqdm

etapas = ["Extração (RPA)", "Carregamento CSV", "Cruzamento Regra de Negócio", "Integração N8N"]
progresso = tqdm(etapas, desc="Pipeline Geral", unit="etapa", position=0, leave=True)

# 1. Extração
progresso.set_description("1/4 Extração (RPA)")
df_clientes = extrair_clientes(URL_CLIENTES)
progresso.update(1)

if df_clientes.empty:
    display(badge("❌ Não foi possível prosseguir: tabela de clientes vazia.", "erro"))
else:
    # 2. Catálogo CSV
    progresso.set_description("2/4 Carregamento CSV")
    df_invest = carregar_investimentos(URL_CSV)
    progresso.update(1)

    # 3. Cruzamento
    progresso.set_description("3/4 Cruzamento de Perfis")
    df_rec = cruzar_perfil_saldo(df_clientes, df_invest)
    progresso.update(1)

    # 4. Envio
    progresso.set_description("4/4 Integração N8N")
    modo = modo_widget.value
    webhook = WEBHOOK_N8N if modo != "demo" else ""
    resultado = enviar_para_n8n(df_rec, webhook, modo_ia=ia_widget.value)
    progresso.update(1)

    progresso.close()

    # Resumo Final em Tabela Estilizada
    display(HTML('<div class="header-azul">🏆 PIPELINE EXECUTADO COM SUCESSO</div>'))
    resumo_df = pd.DataFrame([{
        "Cliente": r["nome"],
        "Perfil": r["perfil"],
        "Saldo": formatar_moeda(r["saldo"]),
        "Nº de Opções Aprovadas": len(r["investimentos"]),
    } for _, r in df_rec.iterrows()])

    display(resumo_df.style.set_properties(**{"text-align": "center", "font-family": "Segoe UI"})
           .bar(subset=["Nº de Opções Aprovadas"], color="#28a745")
           .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}]))

    display(badge(f"🎉 Concluído: {len(df_rec)} clientes atendidos com recomendações personalizadas!", "sucesso"))

--- 
## 📥 Célula 10 (Opcional) — Exportação dos Resultados para CSV

In [ ]:
# ============================================================
# ETAPA 10 (OPCIONAL) | EXPORTAÇÃO DOS RESULTADOS
# ============================================================
if "df_rec" in globals() and not df_rec.empty:
    export_df = df_rec[["nome", "perfil", "saldo", "mensagem"]].copy()
    export_df["saldo"] = export_df["saldo"].apply(formatar_moeda)
    nome_arquivo = "recomendacoes_investimentos.csv"
    export_df.to_csv(nome_arquivo, index=False, encoding="utf-8-sig")

    display(HTML('<div class="header-verde">📥 Download das Recomendações</div>'))
    display(badge(f"📁 Arquivo '{nome_arquivo}' gerado com sucesso!", "sucesso"))

    try:
        from google.colab import files
        files.download(nome_arquivo)
    except ImportError:
        print(f"ℹ️ Arquivo salvo localmente em: {nome_arquivo}")
else:
    display(badge("⚠️ Execute a Célula 9 antes de exportar os resultados.", "aviso"))